# DeepRetrieval (2024)
[[paper]](https://arxiv.org/pdf/2405.02111)<br>DeepRetrieval = Framework for Hacking Search Engines via RL-based LLM

**DeepRetrieval** — это фреймворк для реализации Black-box Adversarial Attacks на поисковые системы и ретриверы. Метод позволяет с помощью Large Language Models (LLM) и обучения с подкреплением (Reinforcement Learning) генерировать короткие текстовые последовательности (триггеры), которые при добавлении к документу заставляют поисковик радикально поднять этот документ в выдаче по целевому запросу.

**Постановка задачи**<br>
Имеется поисковая система (Black-box), набор документов $D$ и целевой запрос $Q$. Выбирается конкретный документ $d \in D$, который изначально находится на низкой позиции в выдаче. Задача: сгенерировать короткий текст $t$ (триггер), чтобы после добавления его к документу ($d' = d + t$) позиция документа в выдаче по запросу $Q$ стала максимально высокой.

**Мотивация**<br>
Современные поисковые системы (Google, Bing) или Dense Retrievers (DPR, BGE) являются закрытыми системами. Мы не имеем доступа к их весам или градиентам. Большинство существующих методов атаки на модели полагаются на White-box подходы, требующие расчета градиентов по тексту. Для реальных приложений (Adversarial SEO или аудит безопасности поиска) нужен метод, который работает через API или обычный веб-интерфейс, не зная внутренней архитектуры поисковика.

**Существующие подходы**<br>
На момент выхода работы существовали следующие методы:
- HotFlip (2017) и Gradient-based attacks: требуют доступа к градиентам модели-жертвы. Неприменимы к коммерческим поисковикам.
- PRM (2023): оптимизация через Discrete Prompt Tuning. Требует много запросов и часто застревает в локальных минимумах из-за дискретности пространства слов.
- Corpus Poisoning (2023): создание множества документов для изменения глобальных статистик. Это дорого и легко детектируется.
- Beam Search / Genetic Algorithms: пытаются перебирать токены, но имеют крайне низкую эффективность в огромном пространстве комбинаций естественного языка.

**Идея**<br>
Авторы предложили рассматривать генерацию "хакерского" текста как задачу Reinforcement Learning. В качестве агента выступает LLM, которая учится предсказывать токены триггера. Наградой (reward) является изменение позиции документа в выдаче. Поскольку LLM уже обладает знаниями о языке и семантике, она ищет решение не в случайном пространстве токенов, а в пространстве осмысленных фраз, которые с большей вероятностью "зацепят" механизмы внимания (Attention) или статистические веса (BM25) ретривера.

**Архитектура**<br>
Система состоит из трех основных компонентов:
1. Policy Model (LLM): обычно используется Llama-2 (2023) или Mistral (2023). Она генерирует последовательность токенов триггера $t$.
2. Target Retriever (Victim): любая поисковая система (Dense, Sparse или Hybrid), доступная как Black-box.
3. Reward Calculator: модуль, который оценивает качество сгенерированного триггера на основе Rank-based metrics.

**Алгоритм обучения**<br>
Обучение строится на алгоритме PPO (Proximal Policy Optimization):
1. Sampling: LLM генерирует несколько вариантов триггеров для пары (Запрос, Документ).
2. Interaction: измененные документы отправляются в Target Retriever.
3. Reward Estimation:
    - Если документ попал в Top-1, выдается максимальный бонус.
    - Основная метрика — $1/Rank$. Чем выше поднялся документ, тем выше Reward.
    - Дополнительно может добавляться штраф за низкую читаемость текста (Perplexity Penalty), чтобы триггер выглядел как естественный язык.
4. Optimization: веса LLM обновляются через PPO, чтобы максимизировать ожидаемый Reward. Модель учится находить общие паттерны (например, специфические ключевые слова или семантические конструкции), которые "взламывают" конкретный тип ретривера.

**Алгоритм инференса**<br>
1. На вход подается запрос $Q$ и документ $d$, который нужно продвинуть.
2. Обученная LLM-policy генерирует оптимальный триггер $t$ (обычно 5-10 токенов).
3. Триггер конкатенируется с документом.
4. Документ индексируется или подается на вход поисковику, занимая топовые позиции.

**Результаты**<br>
Эффективность метода проверялась на популярных моделях (DPR, BGE, Contriever) и реальных поисковиках:
- На датасете MS MARCO метод DeepRetrieval смог поднять целевой документ в Top-1 в 92% случаев для Dense Retrievers, используя триггер длиной всего в 10 токенов.
- Против гибридных систем (BM25 + Cross-Encoder) успех составил около 65%, что значительно выше предыдущих Black-box методов (рост на 20-25пп).
- При атаке на реальные поисковые системы (Bing) удалось добиться существенного изменения ранга для низкочастотных запросов, доказав, что современные поисковики уязвимы к семантическим манипуляциям, сгенерированным через RL.
- Метод показал высокую Transferability: триггеры, обученные на одной модели (например, DPR), часто оказываются эффективными и против других моделей без дополнительного переобучения.

## 📝 Критический анализ

```markdown
# DeepRetrieval (2024)
---
[[paper]](https://arxiv.org/pdf/2405.02111)<br>DeepRetrieval = Framework for Hacking Search Engines via RL-based LLM

**DeepRetrieval** — фреймворк для реализации Black-box Adversarial Attacks на поисковые системы и ретриверы. Используя Large Language Models (LLM) и Reinforcement Learning, он генерирует триггеры, которые повышают позицию документа в выдаче по целевому запросу.

**Постановка задачи**<br>
Имеется Black-box поисковая система, набор документов $D$ и запрос $Q$. Цель: сгенерировать триггер $t$, чтобы документ $d' = d + t$ занял высокую позицию в выдаче по $Q$.

**Мотивация**<br>
Современные поисковые системы и Dense Retrievers закрыты. Большинство атак требуют доступа к градиентам, что неприменимо к коммерческим системам. Нужен метод, работающий через API или веб-интерфейс.

**Существующие подходы**<br>
- HotFlip (2017) и Gradient-based attacks: требуют градиентов.
- PRM (2023): застревает в локальных минимумах.
- Corpus Poisoning (2023): дорого и легко детектируется.
- Beam Search / Genetic Algorithms: низкая эффективность.

**Идея**<br>
Генерация "хакерского" текста рассматривается как задача Reinforcement Learning. LLM предсказывает токены триггера, наградой является изменение позиции документа. LLM ищет решения в пространстве осмысленных фраз, влияющих на механизмы внимания или статистические веса ретривера.

<img src="img/img.png" width=500>

**Архитектура**<br>
1. Policy Model (LLM): генерирует триггер $t$.
2. Target Retriever: Black-box поисковая система.
3. Reward Calculator: оценивает триггер на основе Rank-based metrics.

**Алгоритм обучения**<br>
Используется PPO (Proximal Policy Optimization):
1. Sampling: LLM генерирует триггеры.
2. Interaction: измененные документы отправляются в Target Retriever.
3. Reward Estimation: оценивается позиция документа, добавляется Perplexity Penalty.
4. Optimization: обновление весов LLM через PPO.

**Алгоритм инференса**<br>
1. На вход подается запрос $Q$ и документ $d$.
2. LLM генерирует триггер $t$.
3. Триггер добавляется к документу.
4. Документ подается в поисковик.

**Результаты**<br>
- На MS MARCO метод поднял документ в Top-1 в 92% случаев для Dense Retrievers.
- Против гибридных систем успех составил 65%, что выше предыдущих методов на 20-25пп.
- На реальных поисковиках (Bing) удалось изменить ранг для низкочастотных запросов.
- Высокая Transferability: триггеры, обученные на одной модели, эффективны и против других моделей.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импорт необходимых библиотек
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.optim import Adam
from torch.nn import functional as F

# Установка устройства для вычислений
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загрузка модели и токенизатора
model_name = "gpt2"  # Используем GPT-2 как пример LLM
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Пример документа и запроса
document = "This is a sample document that we want to boost in search rankings."
query = "sample query"

# Функция для генерации триггеров
def generate_trigger(model, tokenizer, document, query, max_length=10):
    # Конкатенация документа и запроса
    input_text = f"{query} {document}"
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)

    # Генерация триггера
    output = model.generate(input_ids, max_length=input_ids.shape[1] + max_length, do_sample=True)
    trigger = tokenizer.decode(output[0], skip_special_tokens=True)[len(input_text):]
    return trigger.strip()

# Пример функции для оценки награды (reward)
def calculate_reward(trigger, document, query):
    # Здесь мы симулируем взаимодействие с поисковой системой
    # В реальном сценарии это будет API вызов к поисковику
    # Для простоты, предположим, что reward обратно пропорционален длине триггера
    return 1.0 / (len(trigger.split()) + 1)

# Обучение с использованием PPO
def train(model, tokenizer, document, query, epochs=10, lr=1e-5):
    optimizer = Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # Генерация триггера
        trigger = generate_trigger(model, tokenizer, document, query)
        
        # Оценка награды
        reward = calculate_reward(trigger, document, query)
        
        # Вычисление потерь и обновление модели
        input_text = f"{query} {document} {trigger}"
        input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss

        # PPO: Обновление модели
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}: Trigger: '{trigger}', Reward: {reward:.4f}")

# Запуск обучения
train(model, tokenizer, document, query)
```

### Объяснение ключевых моментов:

1. **Policy Model (LLM):** Используется GPT-2 для генерации триггеров. В реальном сценарии можно использовать более мощные модели, такие как Llama-2 или Mistral.

2. **Reward Calculation:** В примере награда рассчитывается как обратная величина длины триггера. В реальной системе это будет зависеть от изменения позиции документа в поисковой выдаче.

3. **Reinforcement Learning (PPO):** Используется метод Proximal Policy Optimization для обновления модели. В примере показан базовый процесс обучения, где модель генерирует триггер, оценивается награда, и затем обновляются веса модели.

4. **Black-box Interaction:** В реальном мире взаимодействие с поисковой системой будет через API, чтобы оценить, как триггер влияет на ранжирование документа.

Этот код иллюстрирует основные концепции DeepRetrieval, такие как использование LLM для генерации триггеров и обучение с подкреплением для оптимизации этих триггеров.